In [ ]:
import mpltex
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.cbook import get_sample_data
import pandas as pd
import numpy as np
import random, itertools, math, re, os
import csv
from collections import defaultdict
os.environ["PATH"] = "/Library/TeX/texbin:" + os.environ["PATH"]

plt.rcParams['pgf.texsystem'] = 'pdflatex'
plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

In [ ]:
import subprocess
print("PATH:", os.environ.get("PATH", ""))
print("latex:", subprocess.run(["which", "latex"], capture_output=True, text=True).stdout)

In [ ]:
df = pd.read_csv('../../data/gyroid/convergence_summary.csv')

# 过滤：去掉 xN=20 & fA=0.5 的异常点
df = df[~((df['xN'] == 20) & (df['fA'] == 0.5))]

# 聚合：每个 algorithm 在每个 xN 上的平均 min_fevals
agg = (df.groupby(['algorithm', 'xN'], as_index=False)['min_fevals']
        .mean()
        .rename(columns={'min_fevals': 'fevals_avg'}))

# 全局样式生成器，无线条型
line_style = mpltex.linestyle_generator(lines=[], hollow_styles=[])
line_style_list = [next(line_style) for _ in range(10)]

# Preconditioner 列表和样式映射
base_names = ['etd', 'sd', 'sis']
def get_preconditioner(algo):
    """从算法名提取 preconditioner"""
    algo_lower = algo.lower()
    for base in base_names:
        if algo_lower == base or algo_lower.endswith('_' + base):
            return base
    return None

def style_of(algo):
    """根据 preconditioner 返回样式"""
    prec = get_preconditioner(algo)
    if prec in base_names:
        idx = base_names.index(prec)
        return line_style_list[idx]
    return line_style_list[-1]  # f

In [ ]:
# 1. Base algorithms plot
@mpltex.acs_decorator
def plot_base_fevals():
    fig, ax = plt.subplots(figsize=(3.3, 2.2))
    
    sub = agg[agg['algorithm'].isin(base_names)]
    for algo in sub['algorithm'].unique():
        tmp = sub[sub['algorithm'] == algo].sort_values('xN')
        style = style_of(algo)
        ax.plot(tmp['xN'], tmp['fevals_avg'],
                label=algo.upper(), **style)
    
    ax.set_xlabel(r'$\chi N$', fontsize=10)
    # ax.set_ylabel('Average min fevals', fontsize=10)
    ax.grid(False)
    # set legend in right middle
    ax.legend(loc='upper right', fontsize=9, frameon=False)
    plt.xlim(17.9, 20.1)
    plt.xticks(np.arange(18.0, 20.5, 0.5))
    plt.ylim(160, 330)
    plt.yticks(np.arange(160, 321, 40))
    plt.tight_layout(pad=0.1)
    plt.savefig('base_fevals.svg')
    plt.show()

plot_base_fevals()


In [ ]:
# 2. Anderson Acceleration plot
@mpltex.acs_decorator
def plot_anderson_fevals():
    fig, ax = plt.subplots(figsize=(3.3, 2.2))
    
    sub = agg[agg['algorithm'].str.lower().str.startswith('anderson_')]
    for algo in sub['algorithm'].unique():
        tmp = sub[sub['algorithm'] == algo].sort_values('xN')
        style = style_of(algo)
        ax.plot(tmp['xN'], tmp['fevals_avg'],
                label='AA-' + algo.lower().split('_')[1].upper(), **style)
    
    ax.set_xlabel(r'$\chi N$', fontsize=10)
    # ax.set_ylabel('Average min fevals', fontsize=10)
    ax.grid(False)
    ax.legend(loc='best', fontsize=9, frameon=False)
    plt.xlim(17.9, 20.1)
    plt.xticks(np.arange(18.0, 20.5, 0.5))
    plt.ylim(40, 90)
    plt.yticks(np.arange(40, 91, 10))
    plt.tight_layout(pad=0.1)
    plt.savefig('anderson_fevals.svg')
    plt.show()

plot_anderson_fevals()


In [ ]:
# 3. OACCEL plot
@mpltex.acs_decorator
def plot_oaccel_fevals():
    fig, ax = plt.subplots(figsize=(3.3, 2.2))
    
    sub = agg[agg['algorithm'].str.lower().str.startswith('oaccel_')]
    for algo in sub['algorithm'].unique():
        tmp = sub[sub['algorithm'] == algo].sort_values('xN')
        style = style_of(algo)
        ax.plot(tmp['xN'], tmp['fevals_avg'],
                label='OA-' + algo.lower().split('_')[1].upper(), **style)
    
    ax.set_xlabel(r'$\chi N$', fontsize=10)
    # ax.set_ylabel('Average min fevals', fontsize=10)
    ax.grid(False)
    ax.legend(loc='upper right', fontsize=9, frameon=False)

    plt.xlim(17.9, 20.1)
    plt.xticks(np.arange(18.0, 20.5, 0.5))
    plt.ylim(80, 180)
    plt.yticks(np.arange(80, 181, 20))

    plt.tight_layout(pad=0.1)
    plt.savefig('oaccel_fevals.svg')
    plt.show()

plot_oaccel_fevals()


In [ ]:
# 4. NGMRES plot
@mpltex.acs_decorator
def plot_ngmres_fevals():
    fig, ax = plt.subplots(figsize=(3.3, 2.2))
    
    sub = agg[agg['algorithm'].str.lower().str.startswith('ngmres_')]
    for algo in sub['algorithm'].unique():
        tmp = sub[sub['algorithm'] == algo].sort_values('xN')
        style = style_of(algo)
        ax.plot(tmp['xN'], tmp['fevals_avg'],
                label='NA-' + algo.lower().split('_')[1].upper(), **style)
    
    ax.set_xlabel(r'$\chi N$', fontsize=10)
    # ax.set_ylabel('Average min fevals', fontsize=10)
    ax.grid(False)
    ax.legend(loc='best', fontsize=9, frameon=False)
    plt.xlim(17.9, 20.1)
    plt.xticks(np.arange(18.0, 20.5, 0.5))
    plt.ylim(80, 180)
    plt.yticks(np.arange(80, 181, 20))

    plt.tight_layout(pad=0.1)
    plt.savefig('ngmres_fevals.svg')
    plt.show()

plot_ngmres_fevals()


In [ ]:
# 5. Combined plot - all algorithms on one figure with y-axis from 0 to 300
@mpltex.acs_decorator
def plot_combined_fevals():
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    
    # Marker 按加速算法类型区分
    accel_markers = {
        'base': 'o',      # 圆形
        'anderson': 's',  # 方形
        'oaccel': '^',    # 三角形
        'ngmres': '*',    # 叉形
        # 'ngmres': 'd',    # 菱形 --- IGNORE ---
    }
    
    # Plot Base algorithms
    sub = agg[agg['algorithm'].isin(base_names)]
    for algo in sub['algorithm'].unique():
        tmp = sub[sub['algorithm'] == algo].sort_values('xN')
        style = style_of(algo)
        style['marker'] = accel_markers['base']
        ax.plot(tmp['xN'], tmp['fevals_avg'],
                label=algo.upper(), **style)
    
    # Plot Anderson Acceleration
    sub = agg[agg['algorithm'].str.lower().str.startswith('anderson_')]
    for algo in sub['algorithm'].unique():
        tmp = sub[sub['algorithm'] == algo].sort_values('xN')
        style = style_of(algo)
        style['marker'] = accel_markers['anderson']
        ax.plot(tmp['xN'], tmp['fevals_avg'],
                label='AA-' + algo.lower().split('_')[1].upper(), **style)
    
    # Plot OACCEL
    sub = agg[agg['algorithm'].str.lower().str.startswith('oaccel_')]
    for algo in sub['algorithm'].unique():
        tmp = sub[sub['algorithm'] == algo].sort_values('xN')
        style = style_of(algo)
        style['marker'] = accel_markers['oaccel']
        ax.plot(tmp['xN'], tmp['fevals_avg'],
                label='OA-' + algo.lower().split('_')[1].upper(), **style)
    
    # Plot NGMRES
    sub = agg[agg['algorithm'].str.lower().str.startswith('ngmres_')]
    for algo in sub['algorithm'].unique():
        tmp = sub[sub['algorithm'] == algo].sort_values('xN')
        style = style_of(algo)
        style['marker'] = accel_markers['ngmres']
        ax.plot(tmp['xN'], tmp['fevals_avg'],
                label='NA-' + algo.lower().split('_')[1].upper(), **style)
    
    ax.set_xlabel(r'$\chi N$', fontsize=11)
    ax.set_ylabel(r'$N_{\text{eval}}$', fontsize=11)
    ax.grid(False)

    ax.axhspan(0, 75, facecolor='green', alpha=0.08)
    ax.axhspan(75, 160, facecolor='blue', alpha=0.08)
    ax.axhspan(160, 300, facecolor='red', alpha=0.08)
    
    # Legend 放在图像外部右侧
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), 
              fontsize=9, frameon=False, ncol=1)
    
    plt.xlim(17.9, 20.1)
    plt.xticks(np.arange(18.0, 20.5, 0.5))
    plt.ylim(0, 300)
    plt.yticks(np.arange(0, 301, 50))
    
    # 为外部 legend 预留空间
    plt.tight_layout(rect=[0, 0, 0.78, 1])
    # plt.savefig('fevals_xN.pdf', bbox_inches='tight')
    # plt.savefig('fevals_xN.png', bbox_inches='tight')
    plt.show()

plot_combined_fevals()